In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ==========================================
# 0. AUTHENTICATION (Using Storage Account Key)
# ==========================================
spark.conf.set(
    "",
    ""
)

In [0]:
# ==========================================
# JDBC STREAM SINK FUNCTIONS FOR SYNAPSE
# ==========================================

def write_ohlcv_to_synapse(batch_df, batch_id):
    """Writes the current micro-batch of OHLCV metrics to Synapse via JDBC"""
    if batch_df.count() > 0:
        (
            batch_df.write
            .format("jdbc")
            .option("url", SYNAPSE_JDBC_URL)
            .option("dbtable", "crypto_ohlcv")
            .option("user", SYNAPSE_USER)
            .option("password", SYNAPSE_PASSWORD)
            .mode("append")
            .save()
        )

def write_anomalies_to_synapse(batch_df, batch_id):
    """Writes the current micro-batch of anomalies to Synapse via JDBC"""
    if batch_df.count() > 0:
        (
            batch_df.write
            .format("jdbc")
            .option("url", SYNAPSE_JDBC_URL)
            .option("dbtable", "crypto_anomalies")
            .option("user", SYNAPSE_USER)
            .option("password", SYNAPSE_PASSWORD)
            .mode("append")
            .save()
        )

In [0]:
# ==========================================
# SYNAPSE DATABASE CONNECTION SETTINGS
# ==========================================
SYNAPSE_SERVER = "shrouk-crypto-synapse.sql.azuresynapse.net" 
SYNAPSE_DATABASE = "cryptopool"                        
SYNAPSE_USER = "cryptomaster"
SYNAPSE_PASSWORD = "CryptoData#123"

# Standard JDBC Connection String Format
SYNAPSE_JDBC_URL = f""

In [0]:
# ==========================================
# INGESTION: CONNECT TO AZURE EVENT HUBS
# ==========================================
# Grab these 4 string parameters (Ingestion Manager)
EVENT_HUB_NAMESPACE = "crypto-stream-namespace" 
EVENT_HUB_NAME = "crypto-raw-stream"           
SAS_KEY_NAME = "RootManageSharedAccessKey"              
SAS_KEY_VALUE = ""            

# Construct the Kafka connection profile
EH_SAS_CONNECTION_STRING = f"Endpoint=sb://{EVENT_HUB_NAMESPACE}.servicebus.windows.net/;SharedAccessKeyName={SAS_KEY_NAME};SharedAccessKey={SAS_KEY_VALUE};EntityPath={EVENT_HUB_NAME}"

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": f"{EVENT_HUB_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EVENT_HUB_NAME,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_SAS_CONNECTION_STRING}";',
    "startingOffsets": "latest",
    "failOnDataLoss": "false"
}

# Ingest raw message stream
raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
    .withColumnRenamed("value", "body") 
    .withColumnRenamed("timestamp", "enqueuedTime")
)

In [0]:
# ==========================================
# 1. BRONZE LAYER: INGESTION & METADATA
# ==========================================
bronze_df = raw_stream.select(
    F.col("body").cast("string").alias("raw_json"),
    F.col("enqueuedTime").alias("ingestion_time"),
    F.col("offset").cast("string").alias("eventhub_offset"),
    F.col("partition").cast("string").alias("eventhub_partition")
)

# Write Bronze data stream out
bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://checkpoints@shrouk.dfs.core.windows.net/bronze")
    .start("abfss://bronze@shrouk.dfs.core.windows.net/crypto_bronze")
)

In [0]:
# ==========================================
# 2. SILVER LAYER: PARSE COMBINED @TRADE STRUCTURE
# ==========================================
trade_schema = StructType([
    StructField("stream", StringType(), True),
    StructField("data", StructType([
        StructField("t", LongType(), True),    # Trade ID (Structural sequential tie-breaker)
        StructField("s", StringType(), True),  # Symbol
        StructField("p", StringType(), True),  # Price
        StructField("q", StringType(), True),  # Quantity
        StructField("E", LongType(), True)     # Event Time (ms)
    ]), True)
])

silver_df = (
    bronze_df
    .withColumn("parsed_json", F.from_json(F.col("raw_json"), trade_schema))
    .filter(F.col("parsed_json.data.s").isNotNull())
    .withColumn("symbol", F.col("parsed_json.data.s"))
    .withColumn("price", F.col("parsed_json.data.p").cast(DecimalType(18, 8)))
    .withColumn("quantity", F.col("parsed_json.data.q").cast(DecimalType(18, 8)))
    .withColumn("event_time", (F.col("parsed_json.data.E") / 1000).cast("timestamp"))
    .withColumn("trade_id", F.col("parsed_json.data.t"))
    .select("symbol", "price", "quantity", "event_time", "trade_id")
)

# Write Silver out with mergeSchema active for tracking evolution
silver_query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://checkpoints@shrouk.dfs.core.windows.net/silver")
    .option("mergeSchema", "true") 
    .trigger(processingTime="10 seconds")
    .start("abfss://silver@shrouk.dfs.core.windows.net/crypto_silver")
)

In [0]:
# ==========================================
# 3. GOLD LAYER 1: OHLCV & VOLATILITY METRICS
# ==========================================
# Window metrics calculation with 2-minute latency watermarks
gold_ohlcv_df = (
    silver_df
    .withWatermark("event_time", "2 minutes")
    .groupBy(F.window("event_time", "1 minute", "30 seconds"), "symbol")
    .agg(
        F.min(F.struct("event_time", "trade_id", "price")).getField("price").alias("open_price"),
        F.max("price").alias("high_price"),
        F.min("price").alias("low_price"),
        F.max(F.struct("event_time", "trade_id", "price")).getField("price").alias("close_price"),
        F.sum("quantity").alias("total_volume"),
        F.sum(F.col("price") * F.col("quantity")).alias("total_trade_value"),
        F.count("*").alias("trade_count"),
        F.avg("price").alias("avg_price"),
        F.stddev("price").alias("price_stddev")
    )
    .select(
        "symbol",
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "open_price", "high_price", "low_price", "close_price",
        "total_volume", "total_trade_value", "trade_count",
        "avg_price", "price_stddev"
    )
    .withColumn("loaded_at", F.current_timestamp())
)

# # Write Gold Market Data directly to storage container
# ohlcv_query = (
#     gold_ohlcv_df.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("mergeSchema", "true")
#     .option("checkpointLocation", "abfss://checkpoints@shrouk.dfs.core.windows.net/gold_ohlcv")
#     .start("abfss://gold@shrouk.dfs.core.windows.net/crypto_ohlcv")
# )
# ==========================================
# REWRITTEN OUTPUT: GOLD MARKET DATA
# ==========================================
ohlcv_query = (
    gold_ohlcv_df.writeStream
    .foreachBatch(write_ohlcv_to_synapse) # <--- Using our JDBC function now
    .outputMode("append")
    .option("checkpointLocation", "abfss://checkpoints@shrouk.dfs.core.windows.net/synapse_ohlcv")
    .trigger(processingTime="10 seconds") 
    .start()
)

In [0]:
# ==========================================
# 4. GOLD LAYER 2: ANOMALY ENGINE
# ==========================================
# Process rolling metrics over a 30-minute interval
gold_anomalies_df = (
    silver_df
    .withWatermark("event_time", "2 minutes")
    .groupBy(F.window("event_time", "30 minutes", "1 minute"), "symbol")
    .agg(
        F.max(F.struct("event_time", "trade_id", "price")).getField("price").alias("close_price"),
        F.avg("price").alias("rolling_avg_30m"),
        F.stddev("price").alias("rolling_stddev_30m"),
        F.avg("quantity").alias("avg_trade_size"),
        F.max("quantity").alias("max_trade_size")
    )
    .withColumn("z_score", F.expr(
        "CASE WHEN rolling_stddev_30m = 0 OR rolling_stddev_30m IS NULL THEN 0 ELSE (close_price - rolling_avg_30m) / rolling_stddev_30m END"
    ).cast("decimal(10,4)"))
    .withColumn("price_deviation_pct", F.expr(
        "CASE WHEN rolling_avg_30m = 0 OR rolling_avg_30m IS NULL THEN 0 ELSE ((close_price - rolling_avg_30m) / rolling_avg_30m) * 100 END"
    ).cast("decimal(10,4)"))
    
    # Calculate order sizing ratio for whale alerts
    .withColumn("volume_spike_ratio", F.expr("""
        CASE 
        WHEN avg_trade_size = 0 OR avg_trade_size IS NULL THEN 0 
        ELSE max_trade_size / avg_trade_size 
        END
    """).cast("decimal(10,2)"))
    
    .withColumn("anomaly_type", 
        F.when(F.abs(F.col("z_score")) > 3.0, "EXTREME_PRICE_SPIKE")
         .when(F.abs(F.col("z_score")) > 2.5, "HIGH_PRICE_DEVIATION")
         .when(F.col("volume_spike_ratio") > 5.0, "VOLUME_SPIKE")
         .when(F.col("price_deviation_pct") > 3.0, "PRICE_SURGE_3PCT")
         .when(F.col("price_deviation_pct") < -3.0, "PRICE_DROP_3PCT")
         .otherwise("NORMAL")
    )
    .filter(F.col("anomaly_type") != "NORMAL")
    .select(
        "symbol", 
        F.col("window.start").alias("window_start"), 
        F.col("window.end").alias("window_end"),
        "close_price", "rolling_avg_30m", "rolling_stddev_30m", "z_score", 
        "price_deviation_pct", "volume_spike_ratio", "anomaly_type"
    )
    .withColumn("detection_time", F.current_timestamp())
)

# # Write Outliers directly to storage container
# anomalies_query = (
#     gold_anomalies_df.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("mergeSchema", "true")
#     .option("checkpointLocation", "abfss://checkpoints@shrouk.dfs.core.windows.net/gold_anomalies")
#     .start("abfss://gold@shrouk.dfs.core.windows.net/crypto_anomalies")
# )
# ==========================================
# REWRITTEN OUTPUT: GOLD ANOMALY ENGINE
# ==========================================
anomalies_query = (
    gold_anomalies_df.writeStream
    .foreachBatch(write_anomalies_to_synapse) # <--- Using our JDBC function now
    .outputMode("append")
    .option("checkpointLocation", "abfss://checkpoints@shrouk.dfs.core.windows.net/synapse_anomalies")
    .trigger(processingTime="10 seconds")
    .start()
)

In [0]:
# # Read the Silver table and show the 10 most recent trades
# silver_snapshot = (
#     spark.read.format("delta")
#     .load("abfss://silver@shrouk.dfs.core.windows.net/crypto_silver")
#     .orderBy(F.col("event_time").desc())
# )

# display(silver_snapshot.limit(10))

symbol,price,quantity,event_time,trade_id
ETHUSDT,1728.35000000,0.00300000,2026-06-22T23:58:52.894Z,4153941999
ETHUSDT,1728.35000000,0.00310000,2026-06-22T23:58:52.894Z,4153942005
ETHUSDT,1728.35000000,0.00290000,2026-06-22T23:58:52.894Z,4153942003
ETHUSDT,1728.35000000,0.00300000,2026-06-22T23:58:52.894Z,4153942002
ETHUSDT,1728.35000000,0.04020000,2026-06-22T23:58:52.894Z,4153942007
ETHUSDT,1728.35000000,0.00290000,2026-06-22T23:58:52.894Z,4153942004
ETHUSDT,1728.35000000,0.02410000,2026-06-22T23:58:52.894Z,4153942006
ETHUSDT,1728.35000000,0.00290000,2026-06-22T23:58:52.894Z,4153942000
ETHUSDT,1728.35000000,0.00580000,2026-06-22T23:58:52.894Z,4153942001
BTCUSDT,64021.81000000,0.00008000,2026-06-22T23:58:52.85Z,6432499795


In [0]:
# # Read the Gold OHLCV table and sort by the newest time windows
# gold_ohlcv_snapshot = (
#     spark.read.format("delta")
#     .load("abfss://gold@shrouk.dfs.core.windows.net/crypto_ohlcv")
#     .orderBy(F.col("window_start").desc())
# )

# display(gold_ohlcv_snapshot.limit(10))

symbol,window_start,window_end,open_price,high_price,low_price,close_price,total_volume,total_trade_value,trade_count,avg_price,price_stddev,loaded_at
SOLUSDT,2026-06-22T23:54:30Z,2026-06-22T23:55:30Z,71.96000000,71.99000000,71.96000000,71.99000000,86.10900000,6197.2274000000000000,79,71.977721518987,0.009051047125801313,2026-06-22T23:59:43.386Z
BNBUSDT,2026-06-22T23:54:30Z,2026-06-22T23:55:30Z,590.51000000,590.64000000,590.51000000,590.64000000,2.68100000,1583.4390800000000000,42,590.582380952381,0.04065377454115294,2026-06-22T23:59:43.386Z
ETHUSDT,2026-06-22T23:54:30Z,2026-06-22T23:55:30Z,1728.10000000,1728.36000000,1728.10000000,1728.20000000,25.36300000,43834.4545220000000000,341,1728.251495601173,0.06621127599469342,2026-06-22T23:59:43.386Z
XRPUSDT,2026-06-22T23:54:30Z,2026-06-22T23:55:30Z,1.12970000,1.12990000,1.12960000,1.12990000,2031.60000000,2295.2916700000000000,50,1.129792000000,8.769148144694008E-5,2026-06-22T23:59:43.386Z
BTCUSDT,2026-06-22T23:54:30Z,2026-06-22T23:55:30Z,64017.99000000,64023.99000000,64017.99000000,64023.99000000,0.23874000,15283.9363965000000000,177,64019.817005649718,1.4941634802253014,2026-06-22T23:59:43.386Z
BTCUSDT,2026-06-22T23:54:00Z,2026-06-22T23:55:00Z,64017.99000000,64023.99000000,64017.99000000,64023.99000000,0.23874000,15283.9363965000000000,177,64019.817005649718,1.4941634802253014,2026-06-22T23:58:51.997Z
XRPUSDT,2026-06-22T23:54:00Z,2026-06-22T23:55:00Z,1.12970000,1.12990000,1.12960000,1.12990000,2031.60000000,2295.2916700000000000,50,1.129792000000,8.769148144694008E-5,2026-06-22T23:58:51.997Z
BNBUSDT,2026-06-22T23:54:00Z,2026-06-22T23:55:00Z,590.51000000,590.64000000,590.51000000,590.64000000,2.68100000,1583.4390800000000000,42,590.582380952381,0.04065377454115294,2026-06-22T23:58:51.997Z
ETHUSDT,2026-06-22T23:54:00Z,2026-06-22T23:55:00Z,1728.10000000,1728.36000000,1728.10000000,1728.20000000,25.36300000,43834.4545220000000000,341,1728.251495601173,0.06621127599469342,2026-06-22T23:58:51.997Z
SOLUSDT,2026-06-22T23:54:00Z,2026-06-22T23:55:00Z,71.96000000,71.99000000,71.96000000,71.99000000,86.10900000,6197.2274000000000000,79,71.977721518987,0.009051047125801313,2026-06-22T23:58:51.997Z


In [0]:
# # Read the Gold Anomalies table to check for triggered alerts
# anomalies_snapshot = (
#     spark.read.format("delta")
#     .load("abfss://gold@shrouk.dfs.core.windows.net/crypto_anomalies")
#     .orderBy(F.col("window_start").desc())
# )

# display(anomalies_snapshot.limit(10))

symbol,window_start,window_end,close_price,rolling_avg_30m,rolling_stddev_30m,z_score,price_deviation_pct,volume_spike_ratio,anomaly_type,detection_time
XRPUSDT,2026-06-22T23:26:00Z,2026-06-22T23:56:00Z,1.12990000,1.129247457627,4.974656908509577E-4,1.3117,0.0578,48.67,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
BTCUSDT,2026-06-22T23:26:00Z,2026-06-22T23:56:00Z,64023.99000000,63991.251205821206,24.08018040926625,1.3596,0.0512,174.80,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
BNBUSDT,2026-06-22T23:26:00Z,2026-06-22T23:56:00Z,590.64000000,590.457101449275,0.16140104826248122,1.1332,0.0310,35.53,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
ETHUSDT,2026-06-22T23:26:00Z,2026-06-22T23:56:00Z,1728.20000000,1727.185982800983,0.912696128821626,1.1110,0.0587,58.19,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
SOLUSDT,2026-06-22T23:26:00Z,2026-06-22T23:56:00Z,71.99000000,71.934695121951,0.04623789883787736,1.1961,0.0769,32.12,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
SOLUSDT,2026-06-22T23:25:00Z,2026-06-22T23:55:00Z,71.99000000,71.934695121951,0.04623789883787736,1.1961,0.0769,32.12,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
XRPUSDT,2026-06-22T23:25:00Z,2026-06-22T23:55:00Z,1.12990000,1.129247457627,4.974656908509577E-4,1.3117,0.0578,48.67,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
BNBUSDT,2026-06-22T23:25:00Z,2026-06-22T23:55:00Z,590.64000000,590.457101449275,0.16140104826248122,1.1332,0.0310,35.53,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
BTCUSDT,2026-06-22T23:25:00Z,2026-06-22T23:55:00Z,64023.99000000,63991.251205821206,24.08018040926625,1.3596,0.0512,174.80,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
ETHUSDT,2026-06-22T23:25:00Z,2026-06-22T23:55:00Z,1728.20000000,1727.185982800983,0.912696128821626,1.1110,0.0587,58.19,VOLUME_SPIKE,2026-06-22T23:59:20.044Z
